# TFG — Integración final
### `master_diario.csv` · `master_mensual.csv`

**Archivos necesarios en `/content/`:**
- `infonieve_clean_tfg.csv`
- `orientacion_estaciones.csv`
- `meteo_estaciones_2018_2025.csv`
- `festivos_espana_2018_2026.csv`
- `hoteles_clean.csv`

In [1]:
import pandas as pd
import numpy as np

In [2]:
# ── Carga ──────────────────────────────────────────────────────────────────────
infonieve   = pd.read_csv('/content/infonieve_clean_tfg.csv')
orientacion = pd.read_csv('/content/orientacion_estaciones.csv')
meteo       = pd.read_csv('/content/meteo_estaciones_2018_2025.csv')
festivos    = pd.read_csv('/content/festivos_espana_2018_2026.csv')
hoteles     = pd.read_csv('/content/hoteles_clean.csv', sep=';')

infonieve['fecha'] = pd.to_datetime(infonieve['fecha'])
meteo['fecha']     = pd.to_datetime(meteo['fecha'])
festivos['fecha']  = pd.to_datetime(festivos['fecha'])

print(f'infonieve    {infonieve.shape[0]:>7,} filas × {infonieve.shape[1]} cols')
print(f'orientacion  {orientacion.shape[0]:>7,} filas × {orientacion.shape[1]} cols')
print(f'meteo        {meteo.shape[0]:>7,} filas × {meteo.shape[1]} cols')
print(f'festivos     {festivos.shape[0]:>7,} filas × {festivos.shape[1]} cols')
print(f'hoteles      {hoteles.shape[0]:>7,} filas × {hoteles.shape[1]} cols')

infonieve    142,893 filas × 30 cols
orientacion       30 filas × 7 cols
meteo         14,610 filas × 9 cols
festivos         144 filas × 7 cols
hoteles        1,764 filas × 9 cols


In [3]:
# ── Preparar meteo ─────────────────────────────────────────────────────────────
# meteo cubre 5 estaciones con nombres distintos a infonieve.
# Mapping justificado:
#   Candanchú / Astún → 'Candanchu/Astun'  (misma estación meteorológica)
#   Panticosa         → 'Zona Huesca'       (proxy: mismo valle pirenaico)
# Las 24 estaciones sin cobertura quedarán con NaN en columnas meteo (limitación documentada).

METEO_MAPPING = {
    'Sierra Nevada': 'Sierra Nevada',
    'Cerler':        'Cerler',
    'Formigal':      'Formigal',
    'Candanchú':     'Candanchu/Astun',
    'Astún':         'Candanchu/Astun',
    'Panticosa':     'Zona Huesca',
}

infonieve['_mk'] = infonieve['estacion'].map(METEO_MAPPING)
meteo_j = meteo.rename(columns={'estacion': '_mk'})

n_con = infonieve['_mk'].notna().sum()
print(f'Filas con cobertura meteo: {n_con:,} / {len(infonieve):,} ({n_con/len(infonieve)*100:.1f}%)')
print(f'Estaciones cubiertas: {list(METEO_MAPPING)}')

Filas con cobertura meteo: 28,647 / 142,893 (20.0%)
Estaciones cubiertas: ['Sierra Nevada', 'Cerler', 'Formigal', 'Candanchú', 'Astún', 'Panticosa']


In [4]:
# ── Preparar festivos ──────────────────────────────────────────────────────────
# infonieve ya tiene 'temporada_esqui' → no se recoge de festivos (evita conflicto).
# Se añaden: es_festivo, semana_navidad, semana_santa.

festivos_f = festivos[['fecha', 'semana_navidad', 'semana_santa']].copy()
festivos_f['es_festivo'] = 1

print(f'Fechas con festivo: {len(festivos_f)}')
print(f'Rango: {festivos_f["fecha"].min().date()} → {festivos_f["fecha"].max().date()}')

Fechas con festivo: 144
Rango: 2018-01-01 → 2026-12-26


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# MASTER DIARIO
# Unidad de análisis: (estacion, fecha)
# ══════════════════════════════════════════════════════════════════════════════

# 1. infonieve × orientacion
md = infonieve.merge(orientacion, on='estacion', how='left', validate='many_to_one')
assert md['orientacion'].isna().sum() == 0, 'ERROR: estaciones sin orientacion'
print(f'[1/3] + orientacion → {md.shape}')

# 2. × meteo (LEFT — 6 estaciones con cobertura, resto NaN)
md = md.merge(
    meteo_j[['_mk', 'fecha', 'temp_max_c', 'temp_min_c', 'temp_media_c',
              'precipitacion_mm', 'nieve_cm', 'viento_max_kmh']],
    on=['_mk', 'fecha'], how='left'
)
md.drop(columns=['_mk'], inplace=True)
print(f'[2/3] + meteo      → {md.shape}')

# 3. × festivos (LEFT por fecha)
md = md.merge(festivos_f, on='fecha', how='left')
for col in ['es_festivo', 'semana_navidad', 'semana_santa']:
    md[col] = md[col].fillna(0).astype(int)
print(f'[3/3] + festivos   → {md.shape}')

# Eliminar columna redundante
md.drop(columns=['temporada_inicio'], inplace=True, errors='ignore')

# ── Orden de columnas ──────────────────────────────────────────────────────────
col_order_d = [
    # Identificadores
    'estacion', 'fecha', 'anio', 'mes', 'dia_semana', 'nombre_mes',
    # Estáticas
    'comunidad', 'orientacion', 'altitud_base_m', 'altitud_cima_m', 'desnivel_m', 'umbria',
    # Operación
    'estado_abierta', 'es_temporada_esqui', 'temporada_esqui',
    'remontes_abiertos', 'remontes_total', 'pct_remontes_abiertos',
    'pistas_abiertas',  'pistas_total',    'pct_pistas_abiertas',
    'kilometros_abiertos', 'kilometros_total', 'pct_km_abiertos',
    'espesor_minimo', 'espesor_maximo',
    'tipo_nieve', 'tipo_nieve_limpia',
    'nieve_polvo', 'nieve_dura', 'nieve_humeda',
    'nieve_primavera', 'nieve_artificial', 'nieve_pisada',
    # Contexto temporal
    'es_finde', 'es_festivo', 'semana_navidad', 'semana_santa',
    # Meteo (NaN si estación sin cobertura)
    'temp_max_c', 'temp_min_c', 'temp_media_c',
    'precipitacion_mm', 'nieve_cm', 'viento_max_kmh',
]
col_order_d = [c for c in col_order_d if c in md.columns]
master_diario = md[col_order_d].copy()

print(f'\nmaster_diario → {master_diario.shape[0]:,} filas × {master_diario.shape[1]} columnas')

[1/3] + orientacion → (142893, 37)
[2/3] + meteo      → (142893, 42)
[3/3] + festivos   → (142893, 45)

master_diario → 142,893 filas × 44 columnas


In [6]:
# ── Validación master_diario ───────────────────────────────────────────────────
print('='*52)
print('VALIDACIÓN — master_diario')
print('='*52)
print(f'Dimensiones : {master_diario.shape[0]:,} × {master_diario.shape[1]}')
print(f'Estaciones  : {master_diario["estacion"].nunique()}  (esperado 30)')
print(f'Rango fechas: {master_diario["fecha"].min().date()} → {master_diario["fecha"].max().date()}')

dups = master_diario.duplicated(subset=['estacion','fecha']).sum()
print(f'Duplicados  : {dups}  {"✓" if dups==0 else "⚠ REVISAR"}')

print()
print('Nulos principales:')
for c in ['estado_abierta','orientacion','es_festivo','temp_media_c','precipitacion_mm']:
    if c in master_diario.columns:
        n = master_diario[c].isna().sum()
        print(f'  {c:<22}: {n:>7,}  ({n/len(master_diario)*100:.1f}%)')

VALIDACIÓN — master_diario
Dimensiones : 142,893 × 44
Estaciones  : 30  (esperado 30)
Rango fechas: 2009-11-30 → 2025-09-08
Duplicados  : 0  ✓

Nulos principales:
  estado_abierta        :       0  (0.0%)
  orientacion           :       0  (0.0%)
  es_festivo            :       0  (0.0%)
  temp_media_c          : 126,115  (88.3%)
  precipitacion_mm      : 126,115  (88.3%)


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# MASTER MENSUAL — paso 1: agregación del master_diario
# Se filtra 2018-2025 para alinearse con el rango de hoteles.
# ══════════════════════════════════════════════════════════════════════════════

base = master_diario[
    (master_diario['anio'] >= 2018) & (master_diario['anio'] <= 2025)
].copy()

mm = base.groupby(['estacion', 'anio', 'mes']).agg(

    # Operación
    dias_abierta          = ('estado_abierta',        'sum'),
    dias_totales          = ('estado_abierta',        'count'),
    pct_dias_abierta      = ('estado_abierta',        'mean'),
    km_abiertos_med       = ('kilometros_abiertos',   'mean'),
    pct_remontes_med      = ('pct_remontes_abiertos', 'mean'),
    pct_pistas_med        = ('pct_pistas_abiertas',   'mean'),
    pct_km_med            = ('pct_km_abiertos',       'mean'),
    espesor_max_med       = ('espesor_maximo',        'mean'),
    espesor_min_med       = ('espesor_minimo',        'mean'),

    # Tipos de nieve (días con ese tipo activo en el mes)
    dias_nieve_polvo      = ('nieve_polvo',           'sum'),
    dias_nieve_dura       = ('nieve_dura',            'sum'),
    dias_nieve_artificial = ('nieve_artificial',      'sum'),
    dias_nieve_pisada     = ('nieve_pisada',          'sum'),
    dias_nieve_primavera  = ('nieve_primavera',       'sum'),

    # Contexto
    dias_finde            = ('es_finde',              'sum'),
    dias_festivo          = ('es_festivo',            'sum'),
    dias_semana_navidad   = ('semana_navidad',        'sum'),
    dias_semana_santa     = ('semana_santa',          'sum'),
    dias_temporada_esqui  = ('es_temporada_esqui',    lambda x: (x == True).sum()),

    # Meteo (media mensual; NaN si la estación no tiene cobertura)
    temp_max_med          = ('temp_max_c',            'mean'),
    temp_min_med          = ('temp_min_c',            'mean'),
    temp_media_med        = ('temp_media_c',          'mean'),
    precipitacion_total   = ('precipitacion_mm',      'sum'),
    nieve_total_cm        = ('nieve_cm',              'sum'),
    viento_max_med        = ('viento_max_kmh',        'mean'),

).reset_index()

# Variables estáticas (fijas por estación, se añaden al final)
estaticas = master_diario[
    ['estacion','comunidad','orientacion','altitud_base_m',
     'altitud_cima_m','desnivel_m','umbria']
].drop_duplicates('estacion')
mm = mm.merge(estaticas, on='estacion', how='left')

print(f'Agregación mensual: {mm.shape[0]:,} filas × {mm.shape[1]} cols')
print(f'Estaciones: {mm["estacion"].nunique()}')

Agregación mensual: 2,790 filas × 34 cols
Estaciones: 30


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# MASTER MENSUAL — paso 2: join hoteles
# hoteles usa zonas turísticas INE, no nombres de estación.
# Mapping proxy estación → zona más próxima (limitación documentada).
# Estaciones sin zona asignada → NaN en columnas de ocupación.
# ══════════════════════════════════════════════════════════════════════════════

HOTEL_MAPPING = {
    'Formigal':       'Sallent de Gállego',
    'Panticosa':      'Sallent de Gállego',
    'Astún':          'Jaca',
    'Candanchú':      'Jaca',
    'Cerler':         'Benasque',
    'Baqueira Beret': 'Vielha e Mijaran',
    'La Molina':      'Lleida',
    'Masella':        'Lleida',
    'Boí Taüll':      'Vall de Boí, La',
    'Espot':          'Lleida',
    'Port Ainé':      'Lleida',
    'Port del Comte': 'Lleida',
    'Tavascán':       'Lleida',
    'Vallter':        'Lleida',
    'Sierra Nevada':  'Monachil',
}

# Pivotar hoteles: una fila por (zona, anio, mes), columnas general y fin_de_semana
hot_p = hoteles.pivot_table(
    index=['punto_turistico', 'anio', 'mes'],
    columns='tipo_periodo',
    values='ocupacion_pct',
    aggfunc='first'
).reset_index()
hot_p.columns.name = None
hot_p.rename(columns={
    'punto_turistico': 'zona_hotelera',
    'general':         'ocupacion_general_pct',
    'fin_de_semana':   'ocupacion_finde_pct'
}, inplace=True)

mm['zona_hotelera'] = mm['estacion'].map(HOTEL_MAPPING)
mm = mm.merge(hot_p, on=['zona_hotelera', 'anio', 'mes'], how='left')

sin_hotel = sorted(mm[mm['zona_hotelera'].isna()]['estacion'].unique())
print(f'Estaciones sin zona hotelera ({len(sin_hotel)}): {sin_hotel}')
print(f'Cobertura hotelera: {mm["ocupacion_general_pct"].notna().mean()*100:.1f}% de filas')
print(f'Forma tras hoteles: {mm.shape}')

Estaciones sin zona hotelera (15): ['Alto Campoo', 'Fuentes de Invierno', 'Javalambre', 'La Pinilla', 'Manzaneda', 'Puerto de Navacerrada', 'Punto de Nieve Santa Inés', 'San Isidro', 'Sierra de Béjar', 'Valdelinares', 'Valdesquí', 'Valdezcaray', 'Valgrande-Pajares', 'Vall de Nuria', 'Valle Laciana - Leitariegos']
Cobertura hotelera: 39.3% de filas
Forma tras hoteles: (2790, 37)


In [9]:
# ── Orden de columnas master_mensual ──────────────────────────────────────────
col_order_m = [
    # Identificadores
    'estacion', 'anio', 'mes',
    # Estáticas
    'comunidad', 'orientacion', 'altitud_base_m', 'altitud_cima_m', 'desnivel_m', 'umbria',
    # Operación
    'dias_abierta', 'dias_totales', 'pct_dias_abierta',
    'km_abiertos_med', 'pct_remontes_med', 'pct_pistas_med', 'pct_km_med',
    'espesor_max_med', 'espesor_min_med',
    # Nieve
    'dias_nieve_polvo', 'dias_nieve_dura', 'dias_nieve_artificial',
    'dias_nieve_pisada', 'dias_nieve_primavera',
    # Contexto
    'dias_finde', 'dias_festivo', 'dias_semana_navidad',
    'dias_semana_santa', 'dias_temporada_esqui',
    # Meteo
    'temp_max_med', 'temp_min_med', 'temp_media_med',
    'precipitacion_total', 'nieve_total_cm', 'viento_max_med',
    # Hotelero
    'zona_hotelera', 'ocupacion_general_pct', 'ocupacion_finde_pct',
]
col_order_m = [c for c in col_order_m if c in mm.columns]
master_mensual = mm[col_order_m].copy()

print(f'master_mensual → {master_mensual.shape[0]:,} filas × {master_mensual.shape[1]} columnas')

master_mensual → 2,790 filas × 37 columnas


In [10]:
# ── Validación master_mensual ─────────────────────────────────────────────────
print('='*52)
print('VALIDACIÓN — master_mensual')
print('='*52)
print(f'Dimensiones : {master_mensual.shape[0]:,} × {master_mensual.shape[1]}')
print(f'Estaciones  : {master_mensual["estacion"].nunique()}  (esperado 30)')
print(f'Años        : {sorted(master_mensual["anio"].unique())}')

dups = master_mensual.duplicated(subset=['estacion','anio','mes']).sum()
print(f'Duplicados  : {dups}  {"✓" if dups==0 else "⚠ REVISAR"}')

print()
print('Nulos principales:')
for c in ['pct_dias_abierta','km_abiertos_med','temp_media_med',
          'precipitacion_total','ocupacion_general_pct','ocupacion_finde_pct']:
    if c in master_mensual.columns:
        n = master_mensual[c].isna().sum()
        print(f'  {c:<26}: {n:>5,}  ({n/len(master_mensual)*100:.1f}%)')

print()
print('Cobertura hotelera por estación:')
cob = master_mensual.groupby('estacion')['ocupacion_general_pct'].apply(
    lambda x: 'Con datos' if x.notna().any() else 'Sin datos'
).value_counts()
print(cob.to_string())

VALIDACIÓN — master_mensual
Dimensiones : 2,790 × 37
Estaciones  : 30  (esperado 30)
Años        : [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Duplicados  : 0  ✓

Nulos principales:
  pct_dias_abierta          :     0  (0.0%)
  km_abiertos_med           : 1,786  (64.0%)
  temp_media_med            : 2,232  (80.0%)
  precipitacion_total       :     0  (0.0%)
  ocupacion_general_pct     : 1,693  (60.7%)
  ocupacion_finde_pct       : 1,693  (60.7%)

Cobertura hotelera por estación:
ocupacion_general_pct
Sin datos    15
Con datos    15


In [11]:
# ── Guardar y descargar ───────────────────────────────────────────────────────
master_diario.to_csv('/content/master_diario.csv',   index=False)
master_mensual.to_csv('/content/master_mensual.csv', index=False)

print('Guardado:')
print(f'  master_diario.csv   → {master_diario.shape[0]:,} filas × {master_diario.shape[1]} cols')
print(f'  master_mensual.csv  → {master_mensual.shape[0]:,} filas × {master_mensual.shape[1]} cols')

from google.colab import files
files.download('/content/master_diario.csv')
files.download('/content/master_mensual.csv')

Guardado:
  master_diario.csv   → 142,893 filas × 44 cols
  master_mensual.csv  → 2,790 filas × 37 cols


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>